## Instalación de librerias

In [1]:
import pandas as pd
import json
import gzip
import glob

## Detección de archivos NVD

In [2]:
# Buscar automáticamente todos los archivos NVD en la carpeta raw
rutas_nvd = glob.glob('../data/raw/nvdcve-2.0-*.json.gz')

# Mostrar cuántos archivos encontró
print(f'Se encontraron {len(rutas_nvd)} archivos:')
for ruta in rutas_nvd:
    print(ruta)

Se encontraron 4 archivos:
../data/raw\nvdcve-2.0-2023.json.gz
../data/raw\nvdcve-2.0-2024.json.gz
../data/raw\nvdcve-2.0-2025.json.gz
../data/raw\nvdcve-2.0-2026.json.gz


## Extraer datos de NVD

In [3]:
# Lista donde guardamos los registros
registros = []

for ruta in rutas_nvd:
    print(f'\nProcesando: {ruta}')
    
    with gzip.open(ruta, 'rt', encoding='utf-8') as f:
        data = json.load(f)
    
    # NUEVA estructura NVD 2.0
    items = data['vulnerabilities']
    
    for item in items:
        try:
            cve = item['cve']
            
            cve_id = cve['id']
            descripcion = cve['descriptions'][0]['value']
            fecha = cve['published']
            
            # CVSS (puede estar en diferentes versiones)
            metrics = cve.get('metrics', {})
            
            cvss_data = None
            
            if 'cvssMetricV31' in metrics:
                cvss_data = metrics['cvssMetricV31'][0]['cvssData']
            elif 'cvssMetricV30' in metrics:
                cvss_data = metrics['cvssMetricV30'][0]['cvssData']
            
            if cvss_data:
                score = cvss_data.get('baseScore', None)
                severidad = cvss_data.get('baseSeverity', None)
            else:
                score = None
                severidad = None

            registros.append({
                'cve_id': cve_id,
                'descripcion': descripcion,
                'fecha': fecha,
                'cvss_score': score,
                'severidad': severidad
            })
            
        except Exception as e:
            continue


Procesando: ../data/raw\nvdcve-2.0-2023.json.gz

Procesando: ../data/raw\nvdcve-2.0-2024.json.gz

Procesando: ../data/raw\nvdcve-2.0-2025.json.gz

Procesando: ../data/raw\nvdcve-2.0-2026.json.gz


## Crear dataframe

In [4]:
# Convertir lista a DataFrame
df_cve = pd.DataFrame(registros)

print(f'Total registros: {len(df_cve)}')

df_cve.head()

Total registros: 123802


,cve_id,descripcion,fecha,cvss_score,severidad
0,CVE-2023-0028,Cross-site Scripting (XSS) - Stored in GitHub ...,2023-01-01T01:15:12.627,5.7,MEDIUM
1,CVE-2023-0029,A vulnerability was found in Multilaser RE708 ...,2023-01-01T14:15:09.963,5.3,MEDIUM
2,CVE-2023-22551,"The FTP (aka ""Implementation of a simple FTP c...",2023-01-01T18:15:17.157,7.5,HIGH
3,CVE-2023-22451,Kiwi TCMS is an open source test management sy...,2023-01-02T16:15:11.063,6.5,MEDIUM
4,CVE-2023-22452,kenny2automate is a Discord bot. In the web in...,2023-01-02T20:15:10.163,6.5,MEDIUM


## Limpieza de datos

In [5]:
# Eliminar registros sin ID
df_cve = df_cve.dropna(subset=['cve_id'])

# Convertir fecha a formato datetime
df_cve['fecha'] = pd.to_datetime(df_cve['fecha'], errors='coerce')

# Eliminar duplicados
df_cve = df_cve.drop_duplicates(subset='cve_id')

df_cve.info()

<class 'pandas.DataFrame'>
RangeIndex: 123802 entries, 0 to 123801
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   cve_id       123802 non-null  str           
 1   descripcion  123802 non-null  str           
 2   fecha        123802 non-null  datetime64[us]
 3   cvss_score   116303 non-null  float64       
 4   severidad    116303 non-null  str           
dtypes: datetime64[us](1), float64(1), str(3)
memory usage: 4.7 MB


## Guardar datos CVE

In [6]:
# Guardar dataset procesado
df_cve.to_csv('../data/processed/cve_data.csv', index=False)

print("Guardado correctamente")

Guardado correctamente


## Cargar exploits

In [7]:
# Leer archivo CSV de exploits
df_exploit = pd.read_csv('../data/raw/files_exploits.csv')

df_exploit.head()

,id,file,description,date_published,author,type,platform,port,date_added,date_updated,verified,codes,tags,aliases,screenshot_url,application_url,source_url
0,16929,exploits/aix/dos/16929.rb,AIX Calendar Manager Service Daemon (rpc.cmsd)...,2010-11-11,Metasploit,dos,aix,NaN,2010-11-11,2011-03-06,1,CVE-2009-3699;OSVDB-58726,Metasploit Framework (MSF),NaN,NaN,NaN,http://aix.software.ibm.com/aix/efixes/securit...
1,19046,exploits/aix/dos/19046.txt,AppleShare IP Mail Server 5.0.3 - Buffer Overflow,1999-10-15,Chris Wedgwood,dos,aix,NaN,1999-10-15,2014-01-02,1,CVE-1999-1015;OSVDB-5970,NaN,NaN,NaN,NaN,https://www.securityfocus.com/bid/61/info
2,19049,exploits/aix/dos/19049.txt,BSDI 4.0 tcpmux / inetd - Crash,1998-04-07,Mark Schaefer,dos,aix,NaN,1998-04-07,2014-01-02,1,OSVDB-82889,NaN,NaN,NaN,NaN,https://www.securityfocus.com/bid/66/info
3,33943,exploits/aix/dos/33943.txt,Flussonic Media Server 4.1.25 < 4.3.3 - Arbitr...,2014-07-01,BGA Security,dos,aix,8080.0,2014-07-01,2014-07-01,0,OSVDB-108610;OSVDB-108609,NaN,NaN,NaN,NaN,NaN
4,19418,exploits/aix/dos/19418.txt,IBM AIX 4.3.1 - 'adb' Denial of Service,1999-07-12,GZ Apple,dos,aix,NaN,1999-07-12,2017-11-15,1,OSVDB-83455,NaN,NaN,NaN,NaN,https://www.securityfocus.com/bid/520/info


## Extraer CVE

In [8]:
# Extraer CVE usando regex
df_exploit['cve_id'] = df_exploit['codes'].str.extract(r'(CVE-\d{4}-\d+)')

## Limpieza de datos

In [9]:
# Eliminar nulos
df_exploit = df_exploit.dropna(subset=['cve_id'])

# Seleccionar columnas relevantes
df_exploit = df_exploit[['cve_id', 'description']]

# Eliminar duplicados
df_exploit = df_exploit.drop_duplicates()

df_exploit.head()

,cve_id,description
0,CVE-2009-3699,AIX Calendar Manager Service Daemon (rpc.cmsd)...
1,CVE-1999-1015,AppleShare IP Mail Server 5.0.3 - Buffer Overflow
5,CVE-2003-0087,IBM AIX 4.3.3/5.1/5.2 - 'libIM' Buffer Overflow
9,CVE-2009-4265,PointDev IDEAL Migration - Buffer Overflow (Me...
10,CVE-2014-9349,RobotStats 1.0 - HTML Injection


## Guardar exploits

In [10]:
df_exploit.to_csv('../data/processed/exploit_data.csv', index=False)

print("Guardado correctamente")

Guardado correctamente
